# banknotes_v2: raw ZIP to clean classification source

**Purpose:** keep all readable, correctly annotated, one-banknote representatives from this source after the frozen bbox gate and natural-context crop.

This notebook only cleans one source. It does not balance classes, create final train/dev/test splits, or train a model.


## 1. Imports and global configuration



In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
from PIL import Image
from IPython.display import display
import pandas as pd
import yaml
import math
import re
import io
import hashlib

In [2]:
DATASET_ID = "banknotes_v2"
RAW_ZIP = Path.home() / "Downloads" / "Banknotes-Recognition-main.zip"

OUTPUT_ROOT = Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\banknotes_v2_simple")
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUN_DIR
DATA_OUT = RUN_ROOT / "BANKNOTES_CLEAN"

TARGET_CLASSES = ["10000", "20000", "50000", "100000", "200000", "500000"]
BBOX_MIN_SHORT_SIDE = 120
CONTEXT_RATIO = 0.20

assert RAW_ZIP.is_file(), f"Không tìm thấy ZIP: {RAW_ZIP}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

archive = ZipFile(RAW_ZIP)
members = set(archive.namelist())

print("Raw ZIP :", RAW_ZIP)
print("Output  :", DATA_OUT)

Raw ZIP : C:\Users\Nguyen\Downloads\Banknotes-Recognition-main.zip
Output  : C:\CS231_RAW_NOTEBOOK_RUNS\banknotes_v2_simple\20261026_003633\BANKNOTES_CLEAN


## 2. Open the ZIP, read data.yaml, and inventory the export



In [3]:
config_member = next(name for name in sorted(members) if name.endswith("/data.yaml"))
config = yaml.safe_load(archive.read(config_member).decode("utf-8-sig"))
class_names = config["names"]

class_to_vnd = {}
for class_id, class_name in enumerate(class_names):
    denomination = re.sub(r"\D", "", str(class_name))
    class_to_vnd[class_id] = denomination

class_table = pd.DataFrame([
    {
        "class_id": class_id,
        "raw_class_name": class_names[class_id],
        "denomination": class_to_vnd[class_id],
        "used": class_to_vnd[class_id] in TARGET_CLASSES,
    }
    for class_id in range(len(class_names))
])

display(class_table)

,class_id,raw_class_name,denomination,used
0,0,10 000 VND,10000,True
1,1,100 000 VND,100000,True
2,2,20 000 VND,20000,True
3,3,200 000 VND,200000,True
4,4,50 000 VND,50000,True
5,5,500 000 VND,500000,True


## 3. Helper functions and source metadata



In [4]:
image_members = [
    name for name in sorted(members)
    if "/images/" in name and Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"}
]

def label_for(image_member):
    label_name = image_member.replace("/images/", "/labels/")
    return label_name.rsplit(".", 1)[0] + ".txt"

def source_name(member):
    """Ví dụ: 010000_1_png.rf.abc123.jpg -> 010000_1"""
    name = Path(member).stem
    name = re.sub(r"\.rf\.[0-9a-f]+$", "", name, flags=re.I)
    name = re.sub(r"_(jpg|jpeg|png)$", "", name, flags=re.I)
    return name

def original_split(member):
    return member.split("/images/")[0].split("/")[-1]

def read_one_box(label_member):
    """Trả về 1 bbox hợp lệ; thiếu/sai/nhiều object -> None."""
    if label_member not in members:
        return None

    lines = [
        line.strip()
        for line in archive.read(label_member).decode("utf-8-sig").splitlines()
        if line.strip()
    ]
    if len(lines) != 1:
        return None

    parts = lines[0].split()
    if len(parts) != 5:
        return None

    try:
        class_id = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:])
    except ValueError:
        return None

    if class_id not in class_to_vnd:
        return None
    if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
        return None

    return {
        "raw_label": class_to_vnd[class_id],
        "cx": cx,
        "cy": cy,
        "bw": bw,
        "bh": bh,
    }

print("Số file ảnh export trong ZIP:", len(image_members))

Số file ảnh export trong ZIP: 2122


## 4. Build one valid representative per source



In [5]:
records = []

for member in image_members:
    label_member = label_for(member)
    box = read_one_box(label_member)

    try:
        raw_bytes = archive.read(member)
        with Image.open(io.BytesIO(raw_bytes)) as image:
            image.load()
            width, height = image.size
        readable = True
    except Exception:
        width, height = 0, 0
        readable = False

    valid = readable and box is not None and box["raw_label"] in TARGET_CLASSES

    row = {
        "member": member,
        "annotation_member": label_member,
        "source_name": source_name(member),
        "original_split": original_split(member),
        "width": width,
        "height": height,
        "source_image_sha256": hashlib.sha256(raw_bytes).hexdigest() if readable else "",
        "valid": valid,
    }

    if box is None:
        row.update({"raw_label":"", "cx":0.0, "cy":0.0, "bw":0.0, "bh":0.0})
    else:
        row.update(box)

    records.append(row)

all_images = pd.DataFrame(records)

# Với mỗi source, ưu tiên file hợp lệ; nếu có nhiều file hợp lệ thì lấy file đầu tiên theo tên.
all_images = all_images.sort_values(
    ["source_name", "valid", "member"],
    ascending=[True, False, True]
)

representatives = all_images.drop_duplicates("source_name", keep="first").copy()
pool = representatives[representatives["valid"]].copy().reset_index(drop=True)

print("Exported image files :", len(all_images))
print("Unique source groups :", len(representatives))
print("Valid source groups  :", len(pool))

display(
    pool.groupby("raw_label")
        .size()
        .reindex(TARGET_CLASSES, fill_value=0)
        .to_frame("valid_sources")
)

Exported image files : 2122
Unique source groups : 1336
Valid source groups  : 1329


,valid_sources
raw_label,
10000,285
20000,147
50000,159
100000,307
200000,216
500000,215


## 5. Apply only verified corrections and removals



In [6]:
CORRECT_LABEL_BY_SHA256 = {
    "e4deae347abba4c8abb4521e09ced1c5b90b32f5099c56ce11cdc8139418ebb8": "20000",
    "f199c502478f3082668890f577c59a22028be4ba2a62dacbef143f1dc44b67a5": "20000",
    "5ece5675a4fae9f2d059bd93327b46ec2fc7f5b69bbd9028f27506bb458c58a1": "50000",
    "19480c5e1003cf525e9229bbc9b7ceeeb019eb2f2bc8902f9a6586f59012d94c": "50000",
    "be6554267de2fb1f7ca95648bd3c6f8027e4197b8658d33a30d8697072b08fe7": "100000",
    "d9e18344fe3863cddb2c007a1ac0ffc6d2bda4fb77cc33c2d382c97e33b701f1": "200000",
}
REMOVE_BY_SHA256 = {}

pool["final_label"] = pool["raw_label"]
for source_sha, corrected_label in CORRECT_LABEL_BY_SHA256.items():
    pool.loc[pool["source_image_sha256"] == source_sha, "final_label"] = corrected_label

verified_removals = 0

corrections_found = pool[
    pool["raw_label"] != pool["final_label"]
][["source_name", "raw_label", "final_label"]]

print("Số correction được áp dụng:", len(corrections_found))
display(corrections_found)

Số correction được áp dụng: 6


,source_name,raw_label,final_label
224,020000_147,500000,20000
251,020000_188,50000,20000
305,050000_121,20000,50000
316,050000_139,20000,50000
444,100000_152,50000,100000
753,200000_99,500000,200000


## 6. Apply the frozen bbox threshold



In [7]:
def pixel_bbox(row):
    width = int(row["width"])
    height = int(row["height"])

    left   = row["cx"] - row["bw"] / 2
    top    = row["cy"] - row["bh"] / 2
    right  = row["cx"] + row["bw"] / 2
    bottom = row["cy"] + row["bh"] / 2

    x1 = max(0, math.floor(left * width))
    y1 = max(0, math.floor(top * height))
    x2 = min(width, math.ceil(right * width))
    y2 = min(height, math.ceil(bottom * height))
    return x1, y1, x2, y2

bbox_rows = []
for row in pool.to_dict("records"):
    x1, y1, x2, y2 = pixel_bbox(row)
    bbox_rows.append({
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_x2": x2,
        "bbox_y2": y2,
        "bbox_short_side_px": min(x2 - x1, y2 - y1),
    })

pool = pd.concat([pool.reset_index(drop=True), pd.DataFrame(bbox_rows)], axis=1)

before_scale_filter = len(pool)
pool = pool[pool["bbox_short_side_px"] >= BBOX_MIN_SHORT_SIDE].copy().reset_index(drop=True)

print("Trước bbox filter :", before_scale_filter)
print("Sau bbox >=120 px :", len(pool))
print("Bị loại vì quá nhỏ:", before_scale_filter - len(pool))

display(
    pool.groupby("final_label")
        .size()
        .reindex(TARGET_CLASSES, fill_value=0)
        .to_frame("after_bbox_filter")
)

Trước bbox filter : 1329
Sau bbox >=120 px : 1329
Bị loại vì quá nhỏ: 0


,after_bbox_filter
final_label,
10000,285
20000,147
50000,159
100000,308
200000,217
500000,213


## 7. Crop with 20% context



In [8]:
def context_box(row):
    x1 = int(row["bbox_x1"])
    y1 = int(row["bbox_y1"])
    x2 = int(row["bbox_x2"])
    y2 = int(row["bbox_y2"])

    width = int(row["width"])
    height = int(row["height"])

    box_width = x2 - x1
    box_height = y2 - y1

    crop_x1 = max(0, round(x1 - CONTEXT_RATIO * box_width))
    crop_y1 = max(0, round(y1 - CONTEXT_RATIO * box_height))
    crop_x2 = min(width, round(x2 + CONTEXT_RATIO * box_width))
    crop_y2 = min(height, round(y2 + CONTEXT_RATIO * box_height))

    return crop_x1, crop_y1, crop_x2, crop_y2

## 8. Export kept samples and write the manifest



In [9]:
manifest_rows = []

for row in pool.sort_values(["final_label", "source_name"]).to_dict("records"):
    raw_bytes = archive.read(row["member"])

    with Image.open(io.BytesIO(raw_bytes)) as image:
        image = image.convert("RGB")
        crop_box = context_box(row)
        crop = image.crop(crop_box)

    safe_source = re.sub(r"[^A-Za-z0-9_.-]+", "_", row["source_name"])
    relative_path = f"{row['final_label']}/BANKNOTES_{safe_source}.png"
    target = DATA_OUT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    crop.save(target, format="PNG", optimize=False)

    manifest_rows.append({
        "relative_path": relative_path,
        "dataset_id": "banknotes_v2",
        "source_name": row["source_name"],
        "capture_group": row["source_name"],
        "source_image_sha256": row["source_image_sha256"],
        "final_crop_sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
        "source_archive_member": row["member"],
        "annotation_member": row["annotation_member"],
        "original_split": row["original_split"],
        "raw_label": row["raw_label"],
        "final_label": row["final_label"],
        "source_width": row["width"],
        "source_height": row["height"],
        "bbox_x1": row["bbox_x1"],
        "bbox_y1": row["bbox_y1"],
        "bbox_x2": row["bbox_x2"],
        "bbox_y2": row["bbox_y2"],
        "bbox_short_side_px": row["bbox_short_side_px"],
        "crop_x1": crop_box[0],
        "crop_y1": crop_box[1],
        "crop_x2": crop_box[2],
        "crop_y2": crop_box[3],
        "crop_width": crop.width,
        "crop_height": crop.height,
    })

result = pd.DataFrame(manifest_rows)
result.to_csv(DATA_OUT / "manifest.csv", index=False)

print("Tổng ảnh output:", len(result))
display(
    result.groupby("final_label")
          .size()
          .reindex(TARGET_CLASSES, fill_value=0)
          .to_frame("images")
)

Tổng ảnh output: 1329


,images
final_label,
10000,285
20000,147
50000,159
100000,308
200000,217
500000,213


## 9. Final source-run audit



In [10]:
assert result["relative_path"].is_unique
assert result["source_name"].is_unique
assert set(result["final_label"]).issubset(set(TARGET_CLASSES))
assert (result["bbox_short_side_px"].astype(int) >= BBOX_MIN_SHORT_SIDE).all()
assert len(list(DATA_OUT.rglob("*.png"))) == len(result)

class_counts = (
    result.groupby("final_label")
          .size()
          .reindex(TARGET_CLASSES, fill_value=0)
)

summary_lines = [
    "BANKNOTES V2 SIMPLE PIPELINE",
    "============================",
    f"Raw exported files: {len(all_images)}",
    f"Unique source groups: {len(representatives)}",
    f"Valid source groups before bbox filter: {before_scale_filter}",
    f"Kept after bbox >= BBOX_MIN_SHORT_SIDEpx: {len(result)}",
    "",
    "Class counts:",
]

for label, count in class_counts.items():
    summary_lines.append(f"{label}: {count}")

summary_text = "\n".join(summary_lines)
(RUN_ROOT / "RUN_SUMMARY.txt").write_text(summary_text, encoding="utf-8")

print(summary_text)

BANKNOTES V2 SIMPLE PIPELINE
Raw exported files: 2122
Unique source groups: 1336
Valid source groups before bbox filter: 1329
Kept after bbox >= BBOX_MIN_SHORT_SIDEpx: 1329

Class counts:
10000: 285
20000: 147
50000: 159
100000: 308
200000: 217
500000: 213


In [11]:
# Tất cả các con số dưới đây được tính lại từ manifest và PNG vừa xuất.
manifest = result if "result" in globals() else manifest
label_col = "final_label" if "final_label" in manifest.columns else "denomination"
png_files = list(DATA_OUT.rglob("*.png"))
unreadable_png = 0
for path in png_files:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception:
        unreadable_png += 1

if "before_scale_filter" in globals():
    valid_before_bbox = int(before_scale_filter)
else:
    valid_before_bbox = len(pool)
removed_by_bbox = valid_before_bbox - len(manifest)
class_counts = manifest.groupby(label_col).size().reindex(TARGET_CLASSES, fill_value=0).to_dict()
invalid_labels = sorted(set(manifest[label_col].astype(str)) - set(TARGET_CLASSES))
path_label_mismatch = sum(
    str(row["relative_path"]).replace("\\", "/").split("/")[0] != str(row[label_col])
    for _, row in manifest.iterrows()
)
min_bbox = int(manifest["bbox_short_side_px"].astype(int).min()) if len(manifest) else 0
duplicate_source = int(manifest["source_name"].duplicated().sum())
duplicate_path = int(manifest["relative_path"].duplicated().sum())
relabeled = len(corrections_found) if "corrections_found" in globals() else int(globals().get("verified_relabels", 0))
removed_by_review = int(globals().get("verified_removals", 0))
checks_pass = (
    len(manifest) == len(png_files)
    and unreadable_png == 0
    and duplicate_source == 0
    and duplicate_path == 0
    and invalid_labels == []
    and path_label_mismatch == 0
    and min_bbox >= BBOX_MIN_SHORT_SIDE
)
source_run_status = "PASS" if checks_pass else "HOLD"

summary_lines = [
    f"DATASET_ID = {DATASET_ID}",
    f"RAW_IMAGES = {len(all_images)}",
    f"SOURCE_GROUPS = {len(representatives)}",
    f"VALID_BEFORE_BBOX = {valid_before_bbox}",
    f"VERIFIED_RELABELS = {relabeled}",
    f"VERIFIED_REMOVALS = {removed_by_review}",
    f"BBOX_LT_120_REMOVED = {removed_by_bbox}",
    f"FINAL_KEPT = {len(manifest)}",
    f"FINAL_CLASS_COUNTS = {class_counts}",
    f"MANIFEST_ROWS = {len(manifest)}",
    f"PNG_COUNT = {len(png_files)}",
    f"DUPLICATE_SOURCE_NAME = {duplicate_source}",
    f"DUPLICATE_RELATIVE_PATH = {duplicate_path}",
    f"INVALID_FINAL_LABELS = {invalid_labels}",
    f"PATH_LABEL_MISMATCH = {path_label_mismatch}",
    f"UNREADABLE_EXPORTED_PNG = {unreadable_png}",
    f"MIN_BBOX_SHORT_SIDE_PX = {min_bbox}",
    f"OUTPUT_ROOT = {DATA_OUT}",
    f"SOURCE_RUN_STATUS = {source_run_status}",
]
summary_text = "\n".join(summary_lines)
(RUN_ROOT / "RUN_SUMMARY.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)
assert source_run_status == "PASS", "Source output audit did not pass; preserve run outputs for review."


DATASET_ID = banknotes_v2
RAW_IMAGES = 2122
SOURCE_GROUPS = 1336
VALID_BEFORE_BBOX = 1329
VERIFIED_RELABELS = 6
VERIFIED_REMOVALS = 0
BBOX_LT_120_REMOVED = 0
FINAL_KEPT = 1329
FINAL_CLASS_COUNTS = {'10000': 285, '20000': 147, '50000': 159, '100000': 308, '200000': 217, '500000': 213}
MANIFEST_ROWS = 1329
PNG_COUNT = 1329
DUPLICATE_SOURCE_NAME = 0
DUPLICATE_RELATIVE_PATH = 0
INVALID_FINAL_LABELS = []
PATH_LABEL_MISMATCH = 0
UNREADABLE_EXPORTED_PNG = 0
MIN_BBOX_SHORT_SIDE_PX = 122
OUTPUT_ROOT = C:\CS231_RAW_NOTEBOOK_RUNS\banknotes_v2_simple\20261026_003633\BANKNOTES_CLEAN
SOURCE_RUN_STATUS = PASS
